#### This preprocessing notebook aims to to continue on from the EDA notebooks and adjusting the finding to suit the model ready for training.

In [2]:
#Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import joblib
import numpy as np

In [3]:
path = Path("/Users/stans/eportfolio/eportfolio_Stanley-Southwick/network-anomaly-detector/data/raw").glob("*.csv") # List all CSV files in the raw data directory

# Loop through the CSV to concatenate them into a single DataFrame
dataframes = []
for csv_file in path:
    df = pd.read_csv(csv_file)
    dataframes.append(df)
# Concatenate all DataFrames into one
network_dataset = pd.concat(dataframes, ignore_index=True)

# Strip whitespace from column names
network_dataset.columns = network_dataset.columns.str.strip()

 #### Dataset have now been loaded and concated I will now start working through the findings starting with dropping duplicate rows and columns after confirming the shape is right.

In [4]:
print(network_dataset.head())

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0                22            166                  1                       1   
1             60148             83                  1                       2   
2               123          99947                  1                       1   
3               123          37017                  1                       1   
4                 0      111161336                147                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                            0                            0   
1                            0                            0   
2                           48                           48   
3                           48                           48   
4                            0                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      0                      0            

In [5]:
print(network_dataset.shape)

(2830743, 79)


In [6]:
print(network_dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

In [7]:
#Drop duplicate column and duplicate rows
network_dataset = network_dataset.drop('Fwd Header Length.1', axis=1)
network_dataset = network_dataset.drop_duplicates()

#### Now to double check the removal of duplicates have worked

In [8]:
print(network_dataset.shape)
print('Fwd Header Length.1' in network_dataset.columns)

(2522362, 78)
False


#### The replacement of infinite values with NaN for further preprocessing

In [9]:
inf_before = np.isinf(network_dataset.select_dtypes(include=[float])).sum()
print("Columns with infinite values before replacement:")
print(inf_before[inf_before > 0])

network_dataset = network_dataset.replace([np.inf, -np.inf], np.nan)

is_infinite_float = np.isinf(network_dataset.select_dtypes(include=[float])).sum()
print("\nColumns with infinite values:")
print(is_infinite_float[is_infinite_float > 0])

Columns with infinite values before replacement:
Flow Bytes/s      1211
Flow Packets/s    1564
dtype: int64

Columns with infinite values:
Series([], dtype: int64)


#### Now that we have changed the infinite values into NaN I will look into how much null values are in each column and decide on the strategy

In [10]:
# Check for null values
is_null = network_dataset.isna().sum()
print("\nColumns with null values:")
print(is_null[is_null > 0])

# Drop the null values
network_dataset = network_dataset.dropna()
# Check for null values again
is_null_after = network_dataset.isna().sum()
print("\nColumns with null values after dropping:")
print(is_null_after[is_null_after > 0])

print(network_dataset.shape)


Columns with null values:
Flow Bytes/s      1564
Flow Packets/s    1564
dtype: int64

Columns with null values after dropping:
Series([], dtype: int64)
(2520798, 78)


#### Moving on into the web attacks encoding errors

In [11]:
# Before replacing the encoding errors, let's check the unique values in the 'Label' column
print(network_dataset.Label.unique())

# Replacing the encoding errors with -
network_dataset['Label'] = network_dataset['Label'].replace('Web Attack � Brute Force', 'Web Attack - Brute Force')
network_dataset['Label'] = network_dataset['Label'].replace('Web Attack � XSS', 'Web Attack - XSS')
network_dataset['Label'] = network_dataset['Label'].replace('Web Attack � Sql Injection', 'Web Attack - Sql Injection')


# After replacing the encoding errors, let's check the unique values in the 'Label' column again
print(network_dataset.Label.unique())

['BENIGN' 'Infiltration' 'Bot' 'PortScan' 'DDoS' 'FTP-Patator'
 'SSH-Patator' 'DoS slowloris' 'DoS Slowhttptest' 'DoS Hulk'
 'DoS GoldenEye' 'Heartbleed' 'Web Attack � Brute Force'
 'Web Attack � XSS' 'Web Attack � Sql Injection']
['BENIGN' 'Infiltration' 'Bot' 'PortScan' 'DDoS' 'FTP-Patator'
 'SSH-Patator' 'DoS slowloris' 'DoS Slowhttptest' 'DoS Hulk'
 'DoS GoldenEye' 'Heartbleed' 'Web Attack - Brute Force'
 'Web Attack - XSS' 'Web Attack - Sql Injection']


#### A loop will be created as a solution to iterate through the dataset and remove zero variance columns

In [12]:
no_variance = network_dataset.var(numeric_only=True) == 0
zero_variance_columns = no_variance[no_variance].index.tolist()
print("\nColumns with no variance:")
print(zero_variance_columns)

# Remove columns with no variance
network_dataset = network_dataset.drop(columns=zero_variance_columns)

# Check for columns with no variance again after removal
no_variance_check = network_dataset.var(numeric_only=True) == 0
print("\nColumns with no variance after removal:")
print(no_variance_check[no_variance_check].index.tolist())

print(network_dataset.shape)




Columns with no variance:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Columns with no variance after removal:
[]
(2520798, 70)


### Next step remove highly correlated features 

In [13]:
threshold = 0.95 # Set a threshold for high correlation
correlations = network_dataset.select_dtypes(include='number').corr() # Calculate the correlation matrix for numeric columns
# Shape before dropping highly correlated columns
print(f"\n Shape before dropping highly correlated columns (second of the pairs): {network_dataset.shape}")

# Identify pairs of columns with high correlation
high_corr = (correlations.abs() > threshold) & (correlations.abs() < 1.0)
pairs = [(col, row) for col in correlations.columns 
         for row in correlations.index 
         if high_corr.loc[row, col] and row < col]

columns_to_drop = set()
# Print the pairs of highly correlated columns
for col1, col2 in pairs:
    if col2 not in columns_to_drop:
        columns_to_drop.add(col2)
print("\nColumns to drop due to high correlation:")
print(columns_to_drop)

# Drop the identified columns
network_dataset = network_dataset.drop(columns=columns_to_drop)
# Check the shape after dropping highly correlated columns
print(f"\n Shape after dropping highly correlated columns: {network_dataset.shape}")


 Shape before dropping highly correlated columns (second of the pairs): (2520798, 70)

Columns to drop due to high correlation:
{'ECE Flag Count', 'Bwd Packet Length Max', 'Max Packet Length', 'Fwd IAT Max', 'Subflow Bwd Bytes', 'Flow Packets/s', 'Flow IAT Max', 'Average Packet Size', 'Total Backward Packets', 'Fwd Packet Length Max', 'Flow Duration', 'Idle Mean', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Total Fwd Packets', 'Idle Max', 'Subflow Bwd Packets', 'Avg Bwd Segment Size'}

 Shape after dropping highly correlated columns: (2520798, 52)


#### Starting to encode the categorical, split the data and scale the data

In [14]:
encoder = LabelEncoder()

network_dataset_categorical = network_dataset.dtypes[network_dataset.dtypes == 'object'].index
print(network_dataset_categorical.tolist())

# Encode the categorical columns
for col in network_dataset_categorical:
    network_dataset[col] = encoder.fit_transform(network_dataset[col])

joblib.dump(encoder, '../models/label_encoder.pkl')


['Label']


['../models/label_encoder.pkl']

#### Splitting the data into train test split

In [15]:
X = network_dataset.drop('Label', axis=1) # Features
y = network_dataset['Label'] # Target variable

# Save the preprocessed data to .npy files
np.save('../data/processed/X.npy', X)
np.save('../data/processed/y.npy', y)

# Check the shapes of X and y
print(X.shape)
print(y.shape)
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Check the shapes of the training and testing sets
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Check the distribution of the target variable in the training and testing sets
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(2520798, 51)
(2520798,)
(2016638, 51)
(504160, 51)
(2016638,)
(504160,)
Label
0     0.831109
4     0.068568
2     0.050783
10    0.035978
3     0.004081
7     0.002353
6     0.002136
5     0.002074
11    0.001277
1     0.000773
12    0.000583
14    0.000259
9     0.000014
13    0.000008
8     0.000004
Name: proportion, dtype: float64
Label
0     0.831109
4     0.068568
2     0.050783
10    0.035979
3     0.004080
7     0.002352
6     0.002136
5     0.002075
11    0.001277
1     0.000774
12    0.000583
14    0.000258
9     0.000014
13    0.000008
8     0.000004
Name: proportion, dtype: float64


#### Now to fit standard scaler

In [16]:
network_dataset_scaler = Pipeline([
    ('scaler', StandardScaler())
])

# Fit the scaler on the training data and transform both training and testing data
X_train = network_dataset_scaler.fit_transform(X_train)
X_test = network_dataset_scaler.transform(X_test)
joblib.dump(network_dataset_scaler, '../models/network_dataset_scaler.pkl')

print(X_train.shape)
print(X_test.shape)

(2016638, 51)
(504160, 51)


#### Saving data

In [17]:
import numpy as np
# Save the preprocessed data as .npy files
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)